<a href="https://colab.research.google.com/github/Saurabh07-Nishad/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Saurabh07-Nishad/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

For my Refresh / Content Opportunity Scoring lane, one row represents the daily performance of one content item for one client on one report date. I will use the `fact_content_daily_performance` table. I will develop the analysis on the March 2026 month partition as a mid-panel window, rather than using the final June 2026 month. The data contract will keep the report date explicit because performance signals change over time.

In [15]:
from google.colab import userdata
import requests

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", HF_TOKEN is not None)


HF_TOKEN loaded: True


In [16]:
%pip -q install duckdb

In [17]:
import duckdb

con = duckdb.connect()

print("DuckDB connected successfully")

DuckDB connected successfully


In [18]:
con.execute(
    "CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("Hugging Face access configured")

Hugging Face access configured


In [19]:
REL = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

print("March 2026 partition selected")

March 2026 partition selected


In [20]:
result = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {REL}
""").df()

result

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [21]:
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM {REL}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


## 2. Fields: feature / label / context / excluded

### Features
For my Refresh / Content Opportunity Scoring lane, I will use observable performance signals such as GSC impressions, GSC clicks, GSC CTR, GSC average position, GA4 sessions, and engagement-related metrics when they are available.

### Label / proxy
The future content outcome or performance change will be treated as the label/proxy for scoring. It must be calculated from a later time window and will not be used as an input feature.

### Context
`report_date`, `client_hash_id`, and `content_hash_id` are context fields. They help identify, group, join, and split the data but should not be used as model features.

### Excluded
I will exclude future outcome information and any label-derived fields from the features because they would leak information from the period being predicted. I will also exclude identifiers from the model because they identify entities rather than represent useful content behaviour.

In [22]:
columns = con.sql(f"""
    SELECT *
    FROM {REL}
    LIMIT 0
""").df()

print(columns.columns.tolist())

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


In [23]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Verification plan

I will verify the data contract using four checks:
1. Grain — confirm that each row is unique for report date, client, and content.
2. Counts and date window — measure the number of March 2026 rows and confirm the observed date range.
3. Data availability — check the GSC and GA4 availability flags using `IS TRUE`.
4. Missingness — inspect missing values in the main fields used for the analysis.

In [24]:
grain_check = con.sql(f"""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM {REL}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


In [25]:
count_check = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {REL}
""").df()

count_check

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


In [26]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM {REL}
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows
0,9841378,3611061,413966


In [27]:
missing_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_impressions IS NULL) AS missing_gsc_impressions,
        COUNT(*) FILTER (WHERE gsc_clicks IS NULL) AS missing_gsc_clicks,
        COUNT(*) FILTER (WHERE gsc_avg_position IS NULL) AS missing_gsc_avg_position,
        COUNT(*) FILTER (WHERE ga4_sessions IS NULL) AS missing_ga4_sessions,
        COUNT(*) FILTER (WHERE ga4_engaged_sessions IS NULL) AS missing_ga4_engaged_sessions
    FROM {REL}
""").df()

missing_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,missing_gsc_impressions,missing_gsc_clicks,missing_gsc_avg_position,missing_ga4_sessions,missing_ga4_engaged_sessions
0,9841378,0,0,6230317,3018741,3018741


### Missing-value interpretation

The March 2026 slice has no missing values for GSC impressions or GSC clicks. However, `gsc_avg_position` is missing for 6,230,317 rows, while `ga4_sessions` and `ga4_engaged_sessions` are missing for 3,018,741 rows.

This indicates that availability is not uniform across the dataset. I will not treat missing analytics values as ordinary zeros. GSC and GA4 availability flags will be used to determine whether these metrics are valid for a given row.

In [28]:
window_check = con.sql(f"""
    SELECT
        MIN(report_date) AS first_report_date,
        MAX(report_date) AS last_report_date,
        COUNT(DISTINCT report_date) AS number_of_dates
    FROM {REL}
""").df()

window_check

,first_report_date,last_report_date,number_of_dates
0,2026-03-01,2026-03-31,31


### Verification results

The March 2026 partition contains 9,841,378 observed rows covering 31 report dates from 2026-03-01 to 2026-03-31.

The grain check returned no duplicate combinations of report date, client, and content, supporting the stated daily client-content grain.

GSC data is available for 3,611,061 rows, while GA4 data is available for 413,966 rows. This shows that analytics availability is not uniform across the panel, so availability flags must be respected when using these fields.

## 4. Data limits

The data can show observed performance patterns, but it cannot by itself prove why a content item gained or lost traffic.

The history is unbalanced across clients because different clients have different data-start dates. Therefore, a missing earlier period should not automatically be interpreted as zero performance.

GSC and GA4 availability also differs across rows. The March 2026 checks showed that only 3,611,061 rows had GSC data available and 413,966 rows had GA4 data available. Therefore, metrics from an unavailable source should not be treated as ordinary zero values.

The March 2026 partition is a mid-panel development window. It is useful for checking the data contract and query logic, but one month alone cannot establish a reliable long-term trend.

Finally, the data is observational. It can support directional decision-making and prioritization, but it cannot prove that a specific content change caused a particular performance outcome.

### Named limitation

A key limitation of my slice is uneven data availability across clients and rows, especially for GA4. This means comparisons involving engagement metrics may represent only the subset where those metrics are available.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.